# Rubric grading and revision

`RubricMiddleware` asks a separate grader to evaluate the agent output against explicit criteria. A failed criterion can trigger another attempt. The iteration cap prevents endless self-improvement. A grader verdict is model output, not proof of correctness; inspect the recorded criteria and evidence.

[Rubric grading](https://docs.langchain.com/oss/python/deepagents/rubric)

### Grading, revision and terminal outcomes

The invocation supplies the rubric. A separate grader evaluates each criterion; `needs_revision` injects feedback and returns to the working model. `satisfied` ends successfully. `max_iterations_reached` ends at the revision cap, `failed` means the rubric cannot be evaluated, and `grader_error` records a grading failure.

A grader can use evidence-gathering tools, its own middleware and runtime context. `on_evaluation` receives each verdict and per-criterion results. Grading events also expose iteration starts and ends during streaming. A grader verdict and an independent programmatic check are separate evidence.

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from functools import partial
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from deepagents.middleware.rubric import RubricMiddleware
from lab.custom import check_onboarding, show_evaluation
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from model_config import get_model, run_config
from langchain.tools import tool


In [ ]:
# Execute
@tool
def inspect_onboarding(text: str) -> dict:
    """Count onboarding lines and check handbook/laptop requirements with Python."""
    return check_onboarding(text)

evaluations = []

agent = create_agent(
    model=get_model(),
    system_prompt="Return exactly two non-empty lines, one onboarding step per line. No introduction, heading or closing sentence.",
    middleware=[
        RubricMiddleware(
            model=get_model("grader"), max_iterations=2, tools=[inspect_onboarding],
            grader_middleware=[ModelCallLimitMiddleware(run_limit=3), TraceMiddleware("13_grader", pretty=True)],
            on_evaluation=partial(show_evaluation, history=evaluations),
        ),
        TraceMiddleware("13_rubric", pretty=True),
    ],
    checkpointer=InMemorySaver(),
)
result = agent.invoke(
    {"messages": [("user", prompt("rubric"))], "rubric": prompt("rubric_rules")},
    config=run_config("rubric"),
)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)
print("Verdict path:", [entry["result"] for entry in evaluations])
print("Independent Python checks:", check_onboarding(result["messages"][-1].content))
assert evaluations, "The grader did not produce an evaluation."

The rubric is invocation state rather than an ordinary prompt. Without a rubric, grading is inactive. The callback receives per-criterion evidence and verdicts. A grader can misjudge an answer, so deterministic checks remain independent. Structured grader responses must match the verdict schema.